# AIaaS — Image Generation Throughput Benchmark (Diffusers)

Portable throughput benchmark for the **image generation** workload. Sweeps batch
size (and steps) and reports images/s, seconds/image, and VRAM — VRAM-tiered so
it runs on a T4 (SD-Turbo) or an A100 (SDXL).

Reports per (batch, steps):
- **images/s** and **seconds/image**
- **peak VRAM**

> Tier: *portable proxy* — a clean `diffusers` timing run, not MLPerf
> stable-diffusion-xl (which is accuracy/quality-gated). Good for hardware feel
> and $/image; for leaderboard numbers use the MLPerf SDXL benchmark.

## 1. Install

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "diffusers", "transformers", "accelerate", "pandas"], check=False)
import diffusers
print("diffusers", diffusers.__version__)


## 2. Environment & hardware capture

In [ ]:
import os, json, platform, subprocess, datetime
import torch

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

CUDA = torch.cuda.is_available()
assert CUDA, "No CUDA GPU detected. This benchmark requires a GPU runtime."
_cc = torch.cuda.get_device_capability(0)

ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "compute_capability": f"{_cc[0]}.{_cc[1]}",
    "cuda": torch.version.cuda,
    "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__,
    "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 3. Configuration
VRAM-tiered, ungated models. SDXL on big GPUs; SD-Turbo (small, few-step) on a T4.

In [ ]:
VRAM = ENV["vram_total_gb"]
TIER = "large" if VRAM >= 24 else "small"
if TIER == "large":
    MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
    STEPS = 30
    HEIGHT = WIDTH = 1024
else:
    MODEL = "stabilityai/sd-turbo"   # distilled, 1-4 steps, ~2-3 GB
    STEPS = 4
    HEIGHT = WIDTH = 512

BATCH_SIZES = [1, 2, 4]          # images per generate() call
NUM_ITERS = 3                    # timed iterations per batch size (after warmup)
DTYPE = "bfloat16" if _cc[0] >= 8 else "float16"
PROMPT = "a photo of an astronaut riding a horse on mars, highly detailed"

OUT_DIR = "image_gen_bench_results"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"VRAM={VRAM} GB -> TIER={TIER}, MODEL={MODEL}, steps={STEPS}, "
      f"{WIDTH}x{HEIGHT}, dtype={DTYPE}")


## 4. Load pipeline + run the batch-size sweep

In [ ]:
import time, torch
from diffusers import AutoPipelineForText2Image

dtype = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
pipe = AutoPipelineForText2Image.from_pretrained(MODEL, torch_dtype=dtype)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

def run_bs(bs):
    torch.cuda.reset_peak_memory_stats()
    gen = torch.Generator(device="cuda").manual_seed(0)
    # warmup
    pipe(prompt=[PROMPT] * bs, num_inference_steps=STEPS,
         height=HEIGHT, width=WIDTH, generator=gen)
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(NUM_ITERS):
        pipe(prompt=[PROMPT] * bs, num_inference_steps=STEPS,
             height=HEIGHT, width=WIDTH, generator=gen)
    torch.cuda.synchronize()
    dt = time.time() - t0
    n_images = bs * NUM_ITERS
    return {
        "batch_size": bs,
        "steps": STEPS,
        "images_per_s": round(n_images / dt, 3),
        "s_per_image": round(dt / n_images, 3),
        "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }

RESULTS = []
for bs in BATCH_SIZES:
    try:
        r = run_bs(bs)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        r = {"batch_size": bs, "steps": STEPS, "error": "OOM"}
    print(r)
    RESULTS.append(r)


## 5. Results — normalize, save, summarize

In [ ]:
import pandas as pd
NORM = {
    "schema": "image-gen-bench/1.0",
    "env": ENV, "model": MODEL, "dtype": DTYPE,
    "steps": STEPS, "height": HEIGHT, "width": WIDTH,
    "results": RESULTS,
}
tag = (ENV["platform"] + "_" + ENV["gpu_name"]).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"image_gen_bench_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)
print("Saved:", out)
print(f"\n==== IMAGE-GEN SUMMARY ====\n{ENV['gpu_name']} | {MODEL} | "
      f"{WIDTH}x{HEIGHT} | {STEPS} steps | {DTYPE}")
df = pd.DataFrame(RESULTS)
try:
    from IPython.display import display; display(df)
except Exception:
    print(df.to_string(index=False))


## 6. Reading the result
`images/s` at the largest batch that fits is your generation throughput ceiling;
`s/image` at batch 1 is best-case latency. Feeds a $/image estimate (combine with
power + amortized HW, like the LLM cost model). For credible numbers, MLPerf
stable-diffusion-xl is the next rung.